# Cue Y距離の組合せ最適化 v4
既存700文、品質合格622文、設定C（±0.04、空白0.03）の固定距離帯のみを使用。v3の2方式を基準とし、最大採択数を維持したまま相関・帯間平均差を改善する。比較指標は選択された関連CueのSpearman rhoと5帯のY平均差。探索的な離散最適化であり、大域最適の保証はない。新規API・GPU不要。GitHub専用ブランチに新規保存。


In [ ]:
!pip -q install scipy
import io, json, re, time, base64, hashlib, itertools, unicodedata
from difflib import SequenceMatcher
import numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from scipy.optimize import milp, Bounds, LinearConstraint
from google.colab import userdata
from IPython.display import display
REPO='ryouy/GraduationThesis'
RESULT_BRANCH='experiments/cue-selection-luna-v1'
RUN_ID='20261126_cue_y_optimization_v4'
ROOT='experiments/2026-11/1126_cue_selection_luna/results/'+RUN_ID
SOURCE_REF='328e2c15a0ec64bbbd68112edf95ec7dd31485e3'
SOURCE='experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/pair_probe_scores.csv'
SOURCE_BLOB='c0eebc8cbc5b292cab1cc4f7587fb617f16c60b7'
PAIR_IDS=['C01','C27','C35','C23','C37']
API_MODEL='gpt-6-luna'
EMBED='sentence-transformers/all-mpnet-base-v2'
REV='e8c3b32edf5434bc2275fc9bab85f82640a19130'
STYLES={'structure':'physical structure or characteristic properties','action':'typical action or process','situation':'context or circumstances','function':'purpose, role or effect'}
LEVELS={'very_near':'direct distinctive association','near':'strong common association','medium_near':'association through everyday context','medium_far':'indirect but explainable association','far':'weak but meaningful association','unrelated':'no meaningful association with either concept'}
SETTINGS={'A':(.08,.01),'B':(.06,.02),'C':(.04,.03),'D':(.02,.04)}
Y_WIDTHS=[.05,.075,.10]
MAX_CALLS=400
MAX_ATTEMPTS=4
gh=requests.Session()
gh.headers.update({'Authorization':'Bearer '+userdata.get('GITHUB_RESULTS_TOKEN'),'Accept':'application/vnd.github+json'})
BASE=f'https://api.github.com/repos/{REPO}/contents/'
def dump(o):return (json.dumps(o,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n').encode()
def sha(b):return hashlib.sha256(b).hexdigest()
def blob(b):return hashlib.sha1(b'blob '+str(len(b)).encode()+b'\0'+b).hexdigest()
def get(path,ref=RESULT_BRANCH):
    r=gh.get(BASE+path,params={'ref':ref},timeout=90)
    if r.status_code==404:return None
    r.raise_for_status(); m=r.json()
    if m.get('encoding')=='base64':b=base64.b64decode(m['content'])
    else:
        r=gh.get(BASE+path,params={'ref':ref},headers={'Accept':'application/vnd.github.raw+json'},timeout=90);r.raise_for_status();b=r.content
    assert blob(b)==m['sha']
    return b
def put(name,b):
    path=ROOT+'/'+name
    old=get(path)
    if old is not None:
        if old!=b:raise RuntimeError('Immutable conflict: '+path)
        return
    for i in range(8):
        try:
            r=gh.put(BASE+path,json={'branch':RESULT_BRANCH,'message':'Cue selection: '+name,'content':base64.b64encode(b).decode()},timeout=90)
        except requests.RequestException:
            old=get(path)
            if old is not None:
                if old!=b:raise RuntimeError('Immutable conflict: '+path)
                return
            time.sleep(min(60,2**i))
            continue
        if r.ok:return
        if r.status_code not in (409,422,429,500,502,503,504):r.raise_for_status()
        old=get(path)
        if old is not None:
            if old!=b:raise RuntimeError('Immutable conflict: '+path)
            return
        time.sleep(min(60,2**i))
    raise RuntimeError('GitHub save retries exhausted: '+path)
def save_df(name,d):put(name,d.to_csv(index=False).encode())

INPUT_ROOT='experiments/2026-11/1126_cue_selection_luna/results/20261126_cue_selection_luna_v1'
INPUT_REF=RESULT_BRANCH
raw=get(INPUT_ROOT+'/cue_candidates_measured.csv',INPUT_REF)
assert sha(raw)=='be48364c113d57171e87b4b61bced56b262ec178d2edfe4afb9c888558c2eb1f'
df=pd.read_csv(io.BytesIO(raw))
assert len(df)==700 and df.quality_pass.sum()==622 and df.id.is_unique
assert set(df['style'])==set(STYLES)
assert set(df['pair'])==set(PAIR_IDS)
assert np.isfinite(df[['d_x','d_y']]).all().all()
df[['d_x','d_y']]=df[['d_x','d_y']].astype('float32')
bounds_raw=get(INPUT_ROOT+'/distance_bounds.csv',INPUT_REF)
assert sha(bounds_raw)=='ce0b57df782ef6b64c8bbd4ddbc529bf9dacd817460fff1087f98d7a6b6cd0b4'
old_bounds=pd.read_csv(io.BytesIO(bounds_raw))
def norm(s):return ' '.join(unicodedata.normalize('NFKC',s).casefold().split())

print('Loaded',len(df),'candidates; quality pass',df.quality_pass.sum())

from scipy.stats import rankdata
V3=INPUT_ROOT.replace('luna_v1','corrected_v3')
base_raw=get(V3+'/cue_selected_PROVISIONAL.csv')
assert sha(base_raw)=='018722adbd7077301126130050f3dd3bc3e2e9fba7bd64bdf34c627ca0331d41'
baseline=pd.read_csv(io.BytesIO(base_raw))
base_summary_raw=get(V3+'/selection_summary.csv')
assert sha(base_summary_raw)=='14c28b3e0cf11480f8f0d25f26dcb57f9ba82b91d6f26beefed78ddd282053fb'
base_summary=pd.read_csv(io.BytesIO(base_summary_raw))
SEEDS=[11,23,37,53,71]
STEPS=2500
METHODS=['rho','spread','joint']
config={'input_sha256':sha(raw),'baseline_sha256':sha(base_raw),'baseline_summary_sha256':sha(base_summary_raw),'bounds_sha256':sha(bounds_raw),'setting':'C','seeds':SEEDS,'steps_per_seed':STEPS,'rho_target':.30,'spread_target':.05,'methods':METHODS,'objective':'absolute rho / .30; spread / .05; joint=max(normalized)+0.1*sum(normalized)','status':'exploratory'}
put('config.json',dump(config))



In [ ]:
# Metrics and fixed slot construction. No missing slots may be silently dropped.
def metrics(sel):
    r=sel[sel.band<5]
    x=rankdata(r.d_x);y=rankdata(r.d_y)
    rho=float(np.corrcoef(x,y)[0,1]) if len(r)>2 and np.std(x)>0 and np.std(y)>0 else np.nan
    means=r.groupby('band').d_y.mean()
    spread=float(means.max()-means.min()) if len(means)==5 else np.nan
    return rho,spread,float(r.d_y.std())
def validate(sel,eligible,conflicts,expected):
    assert len(sel)==expected and sel.id.is_unique
    assert not sel.duplicated(['band','style']).any()
    assert set(sel.id)<=set(eligible.id)
    assert sel.quality_pass.all()
    ids=set(sel.id)
    assert not any(a in ids and b in ids for a,b in conflicts)
def prepare(pid):
    g=df[(df.pair==pid)&df.quality_pass].copy()
    bb=old_bounds[(old_bounds.pair==pid)&(old_bounds.setting=='C')&(~old_bounds.y_control)].sort_values('band')
    assert len(bb)==5
    g['band']=-1
    for t in bb.itertuples():
        g.loc[(g.related==1)&g.d_x.between(t.lo,t.hi),'band']=t.band
    g.loc[g.related==0,'band']=5
    g=g[g.band>=0].sort_values('id').reset_index(drop=True)
    slots=list(itertools.product(range(6),STYLES))
    pools=[np.flatnonzero((g.band==b)&(g['style']==s)) for b,s in slots]
    conflict=np.zeros((len(g),len(g)),dtype=bool)
    for i,j in itertools.combinations(range(len(g)),2):
        if SequenceMatcher(None,norm(g.cue.iloc[i]),norm(g.cue.iloc[j])).ratio()>=.80:conflict[i,j]=conflict[j,i]=True
    conflicts=[(g.id.iloc[i],g.id.iloc[j]) for i,j in zip(*np.where(np.triu(conflict,1)))]
    A=np.array([((g.band==b)&(g['style']==s)).astype(float) for b,s in slots])
    extra=[]
    for i,j in zip(*np.where(np.triu(conflict,1))):
        v=np.zeros(len(g));v[i]=v[j]=1;extra.append(v)
    if extra:A=np.vstack([A,extra])
    res=milp(-np.ones(len(g)),integrality=np.ones(len(g)),bounds=Bounds(0,1),constraints=LinearConstraint(A,0,1),options={'time_limit':60})
    assert res.success, res.message
    maximum=int(np.rint(res.x).sum())
    old=baseline[(baseline.pair==pid)&(baseline.setting=='C')&(~baseline.y_control)]
    assert len(old)==maximum
    lookup={x:i for i,x in enumerate(g.id)}
    initial=np.full(24,-1,dtype=int)
    for row in old.itertuples():initial[slots.index((row.band,row.style))]=lookup[row.id]
    validate(g.iloc[initial[initial>=0]],g,conflicts,maximum)
    return g,bb,slots,pools,conflict,conflicts,initial,maximum



In [ ]:
# Seeded simulated annealing: swaps within slots plus moves to currently empty slots.
# Hard constraints: cardinality, quality, distance intervals, styles, duplicate conflicts.
def optimize(g,bb,pools,conflict,initial,method,seed):
    rng=np.random.default_rng(seed)
    dx=g.d_x.to_numpy(dtype=float);dy=g.d_y.to_numpy(dtype=float);band=g.band.to_numpy()
    centers=bb.center.to_numpy()
    def score(state):
        idx=state[state>=0];idx=idx[band[idx]<5]
        if len(idx)<3:return 1e6
        x=rankdata(dx[idx]);y=rankdata(dy[idx])
        x=x-x.mean();y=y-y.mean();den=np.sqrt((x*x).sum()*(y*y).sum())
        rho=abs(float(x@y/den)) if den>0 else 1.
        counts=np.bincount(band[idx],minlength=5)
        sums=np.bincount(band[idx],weights=dy[idx],minlength=5)
        if (counts==0).any():return 1000+100*sum(counts==0)
        means=sums/counts;spread=float(means.max()-means.min())
        a=rho/.30;b=spread/.05
        tie=1e-5*float(np.mean(abs(dx[idx]-centers[band[idx]])))
        return (a if method=='rho' else b if method=='spread' else max(a,b)+.1*(a+b))+tie
    cur=initial.copy();cur_score=score(cur);best=cur.copy();best_score=cur_score
    for step in range(STEPS):
        prop=cur.copy()
        filled=np.flatnonzero(cur>=0)
        empty=np.flatnonzero(cur<0)
        if len(empty) and rng.random()<.20:
            source=int(rng.choice(filled));dest=int(rng.choice(empty))
            if not len(pools[dest]):continue
            prop[source]=-1
        else:
            dest=int(rng.choice(filled))
        if not len(pools[dest]):continue
        candidate=int(rng.choice(pools[dest]))
        prop[dest]=candidate
        ids=prop[prop>=0]
        if conflict[np.ix_(ids,ids)].any():continue
        val=score(prop)
        temp=.5*(.002/.5)**(step/max(STEPS-1,1))
        if val<=cur_score or rng.random()<np.exp(min(0.,(cur_score-val)/temp)):
            cur=prop;cur_score=val
        if val<best_score-1e-12:best=prop.copy();best_score=val
    return best,best_score
results=[];summaries=[];runs=[];missing=[];inventory=[]
for pid in PAIR_IDS:
    g,bb,slots,pools,conflict,conflicts,initial,maximum=prepare(pid)
    for (band,style),pool in zip(slots,pools):
        inventory.append({'pair':pid,'band':band,'style':style,'eligible_n':len(pool),'y_min':float(g.iloc[pool].d_y.min()) if len(pool) else np.nan,'y_max':float(g.iloc[pool].d_y.max()) if len(pool) else np.nan})
    selections={}
    for ctl,label in [(False,'v3_no_y'),(True,'v3_fixed_y')]:
        old=baseline[(baseline.pair==pid)&(baseline.setting=='C')&(baseline.y_control==ctl)]
        selections[label]=g[g.id.isin(old.id)].copy()
        validate(selections[label],g,conflicts,len(old))
    for method in METHODS:
        best=None
        for seed in SEEDS:
            st,score=optimize(g,bb,pools,conflict,initial,method,seed)
            sel=g.iloc[st[st>=0]].copy()
            validate(sel,g,conflicts,maximum)
            rho,spread,sd=metrics(sel)
            runs.append({'pair':pid,'method':method,'seed':seed,'score':score,'rho':rho,'spread':spread,'n':len(sel)})
            if best is None or score<best[0]:best=(score,sel)
        selections[method]=best[1]
    for method,sel in selections.items():
        rho,spread,sd=metrics(sel);nr=int((sel.band<5).sum())
        summaries.append({'pair':pid,'x':g.x.iloc[0],'y':g.y.iloc[0],'method':method,'n':len(sel),'n_related':nr,'maximum_n':maximum,'complete':len(sel)==24,'rho':rho,'abs_rho':abs(rho),'spread':spread,'y_sd':sd,'diagnostic_target_met':bool(abs(rho)<=.30 and spread<=.05),'full_target_met':bool(len(sel)==24 and abs(rho)<=.30 and spread<=.05)})
        results.append(sel.assign(method=method))
        for b,s in slots:
            if not ((sel.band==b)&(sel['style']==s)).any():missing.append({'pair':pid,'method':method,'band':b,'style':s})
    print(pid,'maximum',maximum,'joint',metrics(selections['joint']),flush=True)
summary=pd.DataFrame(summaries);selected=pd.concat(results,ignore_index=True)
assert len(summary)==25
assert not selected.duplicated(['pair','method','band','style']).any()
assert (summary[summary.method.isin(METHODS)].n==summary[summary.method.isin(METHODS)].maximum_n).all()
for name,table in [('summary.csv',summary),('selected_PROVISIONAL.csv',selected),('search_runs.csv',pd.DataFrame(runs)),('missing_slots.csv',pd.DataFrame(missing)),('slot_inventory.csv',pd.DataFrame(inventory))]:save_df(name,table)
display(summary.round(4))
display(summary.groupby('method').agg(slots=('n','sum'),complete_pairs=('complete','sum'),mean_abs_rho=('abs_rho','mean'),mean_spread=('spread','mean'),diagnostic_targets=('diagnostic_target_met','sum'),full_targets=('full_target_met','sum')))



In [ ]:
methods=['v3_no_y','v3_fixed_y']+METHODS
def figsave(name,fig):
    buf=io.BytesIO();fig.savefig(buf,format='png',dpi=150,bbox_inches='tight');put('figures/'+name+'.png',buf.getvalue());plt.show();plt.close(fig)
for field,title,target in [('n','Selected cues',24),('abs_rho','Absolute Spearman rho',.30),('spread','Range of band mean dY',.05)]:
    fig,ax=plt.subplots(figsize=(10,4))
    summary.pivot(index='x',columns='method',values=field).reindex(columns=methods).plot.bar(ax=ax)
    ax.axhline(target,ls='--',color='black');ax.set_ylabel(title);ax.legend(fontsize=8);fig.tight_layout();figsave(field,fig)
for pid in PAIR_IDS:
    fig,axs=plt.subplots(1,5,figsize=(17,3.5),sharex=True,sharey=True)
    g=df[(df.pair==pid)&df.quality_pass&(df.related==1)]
    for ax,method in zip(axs,methods):
        z=selected[(selected.pair==pid)&(selected.method==method)&(selected.band<5)]
        ax.scatter(g.d_x,g.d_y,s=9,color='lightgray')
        ax.scatter(z.d_x,z.d_y,c=z.band,cmap='viridis',vmin=0,vmax=4,s=25)
        ax.set_title(method);ax.set_xlabel('dX')
    axs[0].set_ylabel('dY');fig.suptitle(pid);fig.tight_layout();figsave(pid+'_xy',fig)
files=['summary.csv','selected_PROVISIONAL.csv','search_runs.csv','missing_slots.csv','slot_inventory.csv']
put('manifest.json',dump({'status':'exploratory_not_frozen','config_sha256':sha(dump(config)),'files':{n:sha(get(ROOT+'/'+n)) for n in files}}))
print('https://github.com/'+REPO+'/tree/'+RESULT_BRANCH+'/'+ROOT)



欠損帯を含む部分集合の相関・平均差は参考値。統制方法の比較ではv3固定範囲方式のみ採択数が異なる。相関・平均差を同時に抑える方式は暫定的な目的関数であり、最適解保証はない。本実験前に不足枠の追加生成と独立した品質・Probe文重複監査を行う。
